# Spotify Music Recommendation System

This notebook demonstrates how to use the Spotify Web API to:
- Retrieve trending songs from playlists
- Analyze audio features of songs
- Visualize music characteristics
- Build a personalized recommendation system based on user preferences

We'll use various data science techniques to understand what makes songs popular and how to recommend similar music.

In [ ]:
# Set up user authentication (needed for Development Mode)
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import webbrowser
import pandas as pd
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Spotify API credentials
CLIENT_ID = os.getenv('SPOTIFY_CLIENT_ID')
CLIENT_SECRET = os.getenv('SPOTIFY_CLIENT_SECRET')
REDIRECT_URI = os.getenv('SPOTIFY_REDIRECT_URI', 'http://127.0.0.1:9090/callback')

# Validate that required credentials are present
if not all([CLIENT_ID, CLIENT_SECRET]):
    raise ValueError("Missing required Spotify credentials. Please check your .env file.")

# Define the scopes you need
scope = 'user-library-read user-read-private user-read-email playlist-read-private user-read-playback-state'

# Create the SpotifyOAuth object
sp_oauth = SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI,
    scope=scope,
    open_browser=True
)

# Get the authorization URL
auth_url = sp_oauth.get_authorize_url()
print(f"Please visit this URL to authorize the application: {auth_url}")

# Open the authorization URL in a browser
webbrowser.open(auth_url)

# After you authorize the app, you'll be redirected to your REDIRECT_URI with a code parameter
redirect_url = input("Enter the URL you were redirected to: ")

# Extract the code from the URL and get the token
code = sp_oauth.parse_response_code(redirect_url)
token_info = sp_oauth.get_cached_token()
if not token_info:
    token_info = sp_oauth.get_access_token(code)
user_access_token = token_info['access_token']

print("User access token obtained successfully!")

# Create a Spotify client with the user access token
sp_user = spotipy.Spotify(auth=user_access_token)

# Test the user authentication by getting the current user's profile
try:
    user_profile = sp_user.current_user()
    print(f"Successfully authenticated as {user_profile['display_name']}")
    print(f"User ID: {user_profile['id']}")
    print(f"Email: {user_profile.get('email', 'Email not available')}")
except Exception as e:
    print(f"Authentication test failed: {e}")

In [ ]:
def get_trending_playlist_data_with_user_auth(playlist_id, user_token):
    """Get track data from a Spotify playlist using user authentication"""
    # Initialize the Spotify client with the user access token
    sp = spotipy.Spotify(auth=user_token)
    
    try:
        # Get the tracks from the playlist with proper fields parameter
        playlist_tracks = sp.playlist_tracks(
            playlist_id, 
            fields='items(track(id, name, artists, album(id, name, release_date)))',
            market='IND'  # Specify a market to avoid availability issues
        )

        music_data = []
        for track_info in playlist_tracks['items']:
            if not track_info['track']:
                continue  # Skip tracks that are None (sometimes happens with removed tracks)
                
            track = track_info['track']
            track_name = track.get('name', 'Unknown')
            artists = ', '.join([artist['name'] for artist in track.get('artists', [])])
            album = track.get('album', {})
            album_name = track['album']['name']
            album_id = track['album']['id']
            track_id = track['id']
            release_date = album.get('release_date', None)

            # Get audio features using spotipy's built-in method
            audio_features = None
            if track_id:
                try:
                    audio_features = sp.audio_features([track_id])[0]
                except Exception as e:
                    print(f"Audio features error for track ID {track_id}: {e}")

            # Get track popularity and other metadata
            track_meta = {}
            if track_id:
                try:
                    track_meta = sp.track(track_id)
                except Exception as e:
                    print(f"Track info error for track ID {track_id}: {e}")

            # Append the collected data
            track_data = {
                'Track Name': track_name,
                'Artists': artists,
                'Album Name': album_name,
                'Album ID': album_id,
                'Track ID': track_id,
                'Popularity': track_meta.get('popularity'),
                'Release Date': release_date,
                'Duration (ms)': audio_features.get('duration_ms') if audio_features else None,
                'Explicit': track_meta.get('explicit'),
                'External URLs': track_meta.get('external_urls', {}).get('spotify'),
                'Danceability': audio_features.get('danceability') if audio_features else None,
                'Energy': audio_features.get('energy') if audio_features else None,
                'Key': audio_features.get('key') if audio_features else None,
                'Loudness': audio_features.get('loudness') if audio_features else None,
                'Mode': audio_features.get('mode') if audio_features else None,
                'Speechiness': audio_features.get('speechiness') if audio_features else None,
                'Acousticness': audio_features.get('acousticness') if audio_features else None,
                'Instrumentalness': audio_features.get('instrumentalness') if audio_features else None,
                'Liveness': audio_features.get('liveness') if audio_features else None,
                'Valence': audio_features.get('valence') if audio_features else None,
                'Tempo': audio_features.get('tempo') if audio_features else None,
            }

            music_data.append(track_data)

        return pd.DataFrame(music_data)
    except Exception as e:
        print(f"Error getting playlist data: {e}")
        return pd.DataFrame()  # Return empty DataFrame on error

# Example usage with a popular playlist
# Using "Today's Top Hits" playlist ID
playlist_id = '2AXUnqE0z1oqC0PSm3EqWF'
music_df = get_trending_playlist_data_with_user_auth(playlist_id, user_access_token)

# Display the first few rows of the DataFrame
if not music_df.empty:
    print("\nSuccessfully retrieved playlist data!")
    print(f"Number of tracks: {len(music_df)}")
    print("\nFirst few tracks:")
    print(music_df[['Track Name', 'Artists', 'Popularity']].head())
else:
    print("Failed to retrieve playlist data.")

# Data Analysis and Visualization

Let's analyze and visualize some of the music features to understand the characteristics of trending songs.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if we have data to visualize
if not music_df.empty:
    # Set the style
    sns.set(style="whitegrid")
    
    # Create a figure with multiple subplots
    fig, axs = plt.subplots(2, 2, figsize=(15, 12))
    
    # Plot 1: Distribution of track popularity
    sns.histplot(music_df['Popularity'].dropna(), kde=True, ax=axs[0, 0])
    axs[0, 0].set_title('Distribution of Track Popularity')
    axs[0, 0].set_xlabel('Popularity')
    axs[0, 0].set_ylabel('Count')
    
    # Plot 2: Correlation between danceability and energy
    sns.scatterplot(
        data=music_df,
        x='Danceability',
        y='Energy',
        hue='Popularity',
        size='Popularity',
        sizes=(20, 200),
        palette='viridis',
        ax=axs[0, 1]
    )
    axs[0, 1].set_title('Danceability vs Energy')
    
    # Plot 3: Top 10 most danceable tracks
    top_danceable = music_df.sort_values('Danceability', ascending=False).head(10)
    sns.barplot(
        data=top_danceable,
        y='Track Name',
        x='Danceability',
        palette='Blues_d',
        ax=axs[1, 0]
    )
    axs[1, 0].set_title('Top 10 Most Danceable Tracks')
    axs[1, 0].set_xlabel('Danceability')
    
    # Plot 4: Audio feature correlation heatmap
    features = ['Danceability', 'Energy', 'Loudness', 'Speechiness', 
                'Acousticness', 'Instrumentalness', 'Liveness', 'Valence', 'Tempo']
    correlation = music_df[features].corr()
    sns.heatmap(
        correlation,
        annot=True,
        cmap='coolwarm',
        fmt='.2f',
        ax=axs[1, 1]
    )
    axs[1, 1].set_title('Correlation Between Audio Features')
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization.")

# Personalized Music Recommendations

Using the Spotify Web API, we can generate personalized recommendations based on seed tracks, artists, or genres. This is a powerful feature for creating a recommendation system.

In [ ]:
def get_recommendations(user_access_token, seed_tracks=None, seed_artists=None, seed_genres=None, limit=10):
    """Get Spotify recommendations based on seed tracks, artists, or genres"""
    sp = spotipy.Spotify(auth=user_access_token)
    
    try:
        recommendations = sp.recommendations(
            seed_tracks=seed_tracks or [],
            seed_artists=seed_artists or [],
            seed_genres=seed_genres or [],
            limit=limit
        )
        
        recommended_tracks = []
        for track in recommendations['tracks']:
            track_info = {
                'Track Name': track['name'],
                'Artists': ', '.join([artist['name'] for artist in track['artists']]),
                'Album': track['album']['name'],
                'Popularity': track['popularity'],
                'Preview URL': track['preview_url'],
                'External URL': track['external_urls']['spotify'],
                'Track ID': track['id']
            }
            
            # Get audio features for the track
            audio_features = sp.audio_features(track['id'])[0]
            if audio_features:
                track_info.update({
                    'Danceability': audio_features['danceability'],
                    'Energy': audio_features['energy'],
                    'Acousticness': audio_features['acousticness'],
                    'Valence': audio_features['valence']  # Measures musical positiveness
                })
                
            recommended_tracks.append(track_info)
            
        return pd.DataFrame(recommended_tracks)
    except Exception as e:
        print(f"Error getting recommendations: {e}")
        return pd.DataFrame()


In [ ]:
# Get the top 5 most popular tracks from our original dataset to use as seeds
if not music_df.empty:
    top_tracks = music_df.sort_values('Popularity', ascending=False).head(5)
    seed_track_ids = top_tracks['Track ID'].tolist()
    
    print("Seed tracks being used for recommendations:")
    for i, (name, artist) in enumerate(zip(top_tracks['Track Name'], top_tracks['Artists']), 1):
        print(f"{i}. {name} by {artist}")
    
    # Get recommendations based on these top tracks
    recommendations_df = get_recommendations(user_access_token, seed_tracks=seed_track_ids)
    
    if not recommendations_df.empty:
        print("\nRecommended tracks:")
        display(recommendations_df)
    else:
        print("Could not get recommendations.")
else:
    print("Original dataset is empty, can't get seed tracks.")

# Genre-Based Recommendations

The Spotify API also allows us to get recommendations based on genres. Let's explore this feature.

In [ ]:
# First, let's get available genres that can be used as seeds
def get_available_genres(user_access_token):
    """Get available genre seeds from Spotify with robust error handling"""
    sp = spotipy.Spotify(auth=user_access_token)
    
    try:
        genres = sp.recommendation_genre_seeds()['genres']
        return genres
    except spotipy.exceptions.SpotifyException as e:
        if e.http_status == 404:
            print(f"Error fetching genre seeds: {e}")
            # Provide some common genre seeds as a fallback
            print("Using fallback genre list since the API request failed.")
            return ['pop', 'rap', 'rock', 'electronic', 'dance', 'hip-hop', 
                    'indie', 'jazz', 'classical', 'r-n-b', 'country', 'latin']
        elif e.http_status == 401:
            print(f"Authentication error: {e}")
            print("Your access token may have expired. Try refreshing it.")
            return []
        else:
            print(f"Spotify API error: {e}")
            return []
    except Exception as e:
        print(f"Unexpected error getting genres: {e}")
        return []

available_genres = get_available_genres(user_access_token)
print(f"Spotify has {len(available_genres)} available genres for recommendations.")
print("Sample of available genres:")
print(available_genres[:20])  # Show only the first 20 to keep output manageable

In [ ]:
# Get recommendations based on genres
selected_genres = ['pop', 'dance', 'electronic']  # You can change these based on your preferences

print(f"Getting recommendations for genres: {', '.join(selected_genres)}")
genre_recommendations = get_recommendations(user_access_token, seed_genres=selected_genres)

if not genre_recommendations.empty:
    print("Genre-based recommendations:")
    display(genre_recommendations)
else:
    print("Could not get genre-based recommendations.")

# Artist and Track Analysis

Let's analyze the artists from our dataset and get more details about them.

In [9]:
def get_artist_details(access_token, artist_name):
    """Search for an artist and get their details"""
    sp = spotipy.Spotify(auth=access_token)
    
    try:
        # Search for the artist
        results = sp.search(q=f'artist:{artist_name}', type='artist', limit=1)
        if not results['artists']['items']:
            return None
            
        artist = results['artists']['items'][0]
        
        # Get the artist's top tracks
        top_tracks = sp.artist_top_tracks(artist['id'])
        
        # Get related artists
        related_artists = sp.artist_related_artists(artist['id'])
        
        artist_info = {
            'Name': artist['name'],
            'ID': artist['id'],
            'Popularity': artist['popularity'],
            'Genres': artist['genres'],
            'Followers': artist['followers']['total'],
            'Image URL': artist['images'][0]['url'] if artist['images'] else None,
            'Top Tracks': [track['name'] for track in top_tracks['tracks']],
            'Related Artists': [related['name'] for related in related_artists['artists']]
        }
        
        return artist_info
    except Exception as e:
        print(f"Error getting artist details: {e}")
        return None


In [ ]:
# Get the most common artists in our dataset
if not music_df.empty:
    # Extract the first artist from each track (primary artist)
    music_df['Primary Artist'] = music_df['Artists'].apply(lambda x: x.split(', ')[0] if isinstance(x, str) else '')
    
    # Count the occurrences of each artist
    artist_counts = music_df['Primary Artist'].value_counts().head(5)
    
    print("Top 5 artists in the dataset:")
    display(artist_counts)
    
    # Get details for the top artist
    if not artist_counts.empty:
        top_artist = artist_counts.index[0]
        print(f"\nGetting details for top artist: {top_artist}")
        
        artist_details = get_artist_details(user_access_token, top_artist)
        if artist_details:
            print(f"Artist: {artist_details['Name']}")
            print(f"Popularity: {artist_details['Popularity']}")
            print(f"Genres: {', '.join(artist_details['Genres'])}")
            print(f"Followers: {artist_details['Followers']:,}")
            
            print("\nTop Tracks:")
            for i, track in enumerate(artist_details['Top Tracks'][:5], 1):
                print(f"{i}. {track}")
                
            print("\nRelated Artists:")
            for i, artist in enumerate(artist_details['Related Artists'][:5], 1):
                print(f"{i}. {artist}")
        else:
            print(f"Could not find details for {top_artist}")
else:
    print("Original dataset is empty, can't analyze artists.")

# Building a Simple Recommendation Model

Let's create a basic recommendation model using the audio features from our dataset.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def build_recommendation_model(music_df):
    """Build a simple recommendation model based on audio features"""
    if music_df.empty:
        print("Dataset is empty, can't build model.")
        return None
    
    # Select relevant features for recommendation
    features = ['Danceability', 'Energy', 'Acousticness', 'Valence', 'Speechiness', 'Instrumentalness']
    
    # Check if we have these features in our dataset
    missing_features = [f for f in features if f not in music_df.columns]
    if missing_features:
        print(f"Missing features in dataset: {missing_features}")
        # Filter to only include available features
        features = [f for f in features if f in music_df.columns]
    
    if not features:
        print("No audio features available in dataset.")
        return None
    
    # Drop rows with NaN values in the selected features
    music_features_df = music_df.dropna(subset=features)
    
    if music_features_df.empty:
        print("No complete feature data available.")
        return None
    
    print(f"Building model with {len(music_features_df)} tracks and {len(features)} features.")
    
    # Normalize the features
    scaler = MinMaxScaler()
    features_scaled = scaler.fit_transform(music_features_df[features])
    
    # Create a similarity matrix
    similarity_matrix = cosine_similarity(features_scaled)
    
    return {
        'similarity_matrix': similarity_matrix,
        'music_df': music_features_df.reset_index(drop=True),
        'features': features
    }

def get_similar_tracks(model, track_index, n=5):
    """Get similar tracks based on the similarity matrix"""
    if not model or 'similarity_matrix' not in model:
        return []
    
    # Get similarity scores for the track
    similarity_scores = model['similarity_matrix'][track_index]
    
    # Get indices of most similar tracks (excluding itself)
    similar_indices = np.argsort(similarity_scores)[::-1][1:n+1]
    
    # Get the similar tracks
    similar_tracks = model['music_df'].iloc[similar_indices]
    
    return similar_tracks

# Build the model
recommendation_model = build_recommendation_model(music_df)

if recommendation_model:
    # Get a random track to generate recommendations for
    track_index = np.random.randint(0, len(recommendation_model['music_df']))
    track = recommendation_model['music_df'].iloc[track_index]
    
    print(f"\nGenerating recommendations based on:\n{track['Track Name']} by {track['Artists']}")
    print(f"Features: {', '.join([f'{f}: {track[f]:.2f}' for f in recommendation_model['features']])}")
    
    # Get similar tracks
    similar_tracks = get_similar_tracks(recommendation_model, track_index)
    
    print("\nRecommended similar tracks:")
    for i, (_, similar) in enumerate(similar_tracks.iterrows(), 1):
        print(f"{i}. {similar['Track Name']} by {similar['Artists']}")
        print(f"   Features: {', '.join([f'{f}: {similar[f]:.2f}' for f in recommendation_model['features']])}")
        print(f"   Similarity: {recommendation_model['similarity_matrix'][track_index][similar.name]:.2f}")
        print()

# Time-Weighted Popularity for Latest Releases

To give more emphasis to recent releases in our recommendations, we'll implement a weighted popularity function based on release date.

In [ ]:
from datetime import datetime

# Function to calculate weighted popularity scores based on release date
def calculate_weighted_popularity(release_date):
    # Convert the release date to datetime object
    # Handle different date formats (some Spotify dates are just year or year-month)
    try:
        if len(release_date) == 4:  # Just year
            release_date = datetime.strptime(release_date, '%Y')
        elif len(release_date) == 7:  # Year-month
            release_date = datetime.strptime(release_date, '%Y-%m')
        else:  # Full date
            release_date = datetime.strptime(release_date, '%Y-%m-%d')
    except ValueError as e:
        print(f"Error parsing date {release_date}: {e}")
        # Default to older date if parsing fails
        release_date = datetime(2000, 1, 1)

    # Calculate the time span between release date and today's date
    time_span = datetime.now() - release_date

    # Calculate the weighted popularity score based on time span (e.g., more recent releases have higher weight)
    weight = 1 / (time_span.days + 1)
    return weight

# Test the function with a sample date
sample_date = "2023-04-15"
weight = calculate_weighted_popularity(sample_date)
print(f"Release date: {sample_date}")
print(f"Time weight: {weight:.6f}")

# Calculate time weight for a very recent release
recent_date = "2025-04-01"  # Very recent as of our current date (April 12, 2025)
recent_weight = calculate_weighted_popularity(recent_date)
print(f"Recent release date: {recent_date}")
print(f"Recent time weight: {recent_weight:.6f}")

# Calculate time weight for an older release
old_date = "2010-01-01"
old_weight = calculate_weighted_popularity(old_date)
print(f"Older release date: {old_date}")
print(f"Older time weight: {old_weight:.6f}")

# Content-Based Recommendations Using Audio Features

Let's implement content-based filtering to recommend songs based on their audio characteristics.

In [13]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

# Normalize the music features using Min-Max scaling
def normalize_features(music_df):
    # Select features for normalization
    features = ['Danceability', 'Energy', 'Acousticness', 'Valence', 'Speechiness', 'Instrumentalness']
    
    # Check which features exist in the dataframe
    available_features = [f for f in features if f in music_df.columns]
    
    if not available_features:
        print("No audio features available for normalization.")
        return None, []
    
    # Drop rows with missing values in the features we need
    music_features_df = music_df.dropna(subset=available_features)
    
    if music_features_df.empty:
        print("No complete feature data available.")
        return None, []
    
    # Extract features for scaling
    music_features = music_features_df[available_features].values
    
    # Scale the features
    scaler = MinMaxScaler()
    music_features_scaled = scaler.fit_transform(music_features)
    
    return music_features_scaled, music_features_df.reset_index(drop=True)

# Function to get content-based recommendations based on music features
def content_based_recommendations(input_song_name, music_df, music_features_scaled, num_recommendations=5):
    if input_song_name not in music_df['Track Name'].values:
        print(f"'{input_song_name}' not found in the dataset. Please enter a valid song name.")
        return pd.DataFrame()

    # Get the index of the input song in the music DataFrame
    input_song_index = music_df[music_df['Track Name'] == input_song_name].index[0]

    # Calculate the similarity scores based on music features (cosine similarity)
    similarity_scores = cosine_similarity([music_features_scaled[input_song_index]], music_features_scaled)

    # Get the indices of the most similar songs
    similar_song_indices = similarity_scores.argsort()[0][::-1][1:num_recommendations + 1]

    # Get the names of the most similar songs based on content-based filtering
    columns_to_include = ['Track Name', 'Artists', 'Album Name', 'Release Date', 'Popularity', 'Track ID']
    # Make sure to only include columns that exist in the dataframe
    available_columns = [col for col in columns_to_include if col in music_df.columns]
    content_based_recommendations = music_df.iloc[similar_song_indices][available_columns]

    return content_based_recommendations

# Hybrid Recommendation System

Now we'll combine content-based filtering with time-weighted popularity for a hybrid approach.

In [14]:
def hybrid_recommendations(input_song_name, music_df, music_features_scaled, num_recommendations=5, alpha=0.5):
    """Generate hybrid recommendations based on content similarity and time-weighted popularity"""
    if input_song_name not in music_df['Track Name'].values:
        print(f"'{input_song_name}' not found in the dataset. Please enter a valid song name.")
        return pd.DataFrame()

    # Get content-based recommendations
    content_based_rec = content_based_recommendations(input_song_name, music_df, music_features_scaled, num_recommendations)
    
    if content_based_rec.empty:
        print("Could not generate content-based recommendations.")
        return pd.DataFrame()
    
    # Get the original song's details
    input_song_details = music_df[music_df['Track Name'] == input_song_name]
    
    # Get original popularity score
    if 'Popularity' in input_song_details.columns:
        popularity_score = input_song_details['Popularity'].values[0]
    else:
        popularity_score = 50  # Default if missing
        
    # Get release date and calculate time-weighted popularity
    if 'Release Date' in input_song_details.columns and not pd.isna(input_song_details['Release Date'].values[0]):
        release_date = input_song_details['Release Date'].values[0]
        weighted_popularity_score = popularity_score * calculate_weighted_popularity(release_date)
    else:
        weighted_popularity_score = popularity_score  # Default to original popularity if date is missing
    
    # Add weighted popularity scores to content-based recommendations
    hybrid_recommendations = content_based_rec.copy()
    
    # Calculate weighted popularity for each recommendation
    if 'Release Date' in hybrid_recommendations.columns and 'Popularity' in hybrid_recommendations.columns:
        hybrid_recommendations['Time Weight'] = hybrid_recommendations['Release Date'].apply(
            lambda date: calculate_weighted_popularity(date) if not pd.isna(date) else 0
        )
        hybrid_recommendations['Weighted Popularity'] = hybrid_recommendations['Popularity'] * hybrid_recommendations['Time Weight']
        
        # Calculate the hybrid score (alpha for content similarity, 1-alpha for weighted popularity)
        # Note: Since we don't directly have the similarity scores here, we're using the order as a proxy
        hybrid_recommendations['Content Score'] = [1.0 - (i/num_recommendations) for i in range(len(hybrid_recommendations))]
        # Normalize weighted popularity to 0-1 range
        max_weighted_pop = hybrid_recommendations['Weighted Popularity'].max()
        if max_weighted_pop > 0:
            hybrid_recommendations['Normalized Weighted Pop'] = hybrid_recommendations['Weighted Popularity'] / max_weighted_pop
        else:
            hybrid_recommendations['Normalized Weighted Pop'] = 0
            
        # Compute final hybrid score
        hybrid_recommendations['Hybrid Score'] = (alpha * hybrid_recommendations['Content Score'] + 
                                               (1-alpha) * hybrid_recommendations['Normalized Weighted Pop'])
        
        # Sort by hybrid score
        hybrid_recommendations = hybrid_recommendations.sort_values(by='Hybrid Score', ascending=False)
    
    # Remove the input song from recommendations if it's there
    hybrid_recommendations = hybrid_recommendations[hybrid_recommendations['Track Name'] != input_song_name]
    
    return hybrid_recommendations.head(num_recommendations)

In [ ]:
# Test our hybrid recommendation system with a sample song
if not music_df.empty:
    # Prepare the data for recommendations
    music_features_scaled, filtered_music_df = normalize_features(music_df)
    
    if music_features_scaled is not None:
        # Get a sample track to use for recommendations
        sample_track = filtered_music_df['Track Name'].iloc[0]
        print(f"Generating recommendations for sample track: '{sample_track}'")
        
        # Get pure content-based recommendations
        print("\nContent-based recommendations:")
        content_recs = content_based_recommendations(sample_track, filtered_music_df, music_features_scaled)
        display(content_recs)
        
        # Get hybrid recommendations
        print("\nHybrid recommendations (factoring in release date):")
        hybrid_recs = hybrid_recommendations(sample_track, filtered_music_df, music_features_scaled)
        display(hybrid_recs[[col for col in hybrid_recs.columns if col not in ['Time Weight', 'Content Score', 'Normalized Weighted Pop']]])
        
        # Let the user try with their own track
        print("\nTry with your own track:")
        print("Available tracks in dataset:")
        sample_tracks = filtered_music_df['Track Name'].sample(min(5, len(filtered_music_df))).tolist()
        for i, track in enumerate(sample_tracks, 1):
            print(f"{i}. {track}")
    else:
        print("Could not normalize music features for recommendations.")
else:
    print("No music data available for recommendations.")

# Interactive Track Recommendation

Enter a track name to get personalized recommendations:

In [ ]:
def get_recommendations_for_track(track_name, alpha=0.5, num_recs=5):
    """Get recommendations for a user-specified track"""
    if not music_df.empty:
        # Prepare the data for recommendations
        music_features_scaled, filtered_music_df = normalize_features(music_df)
        
        if music_features_scaled is not None:
            if track_name in filtered_music_df['Track Name'].values:
                print(f"Generating recommendations for track: '{track_name}'")
                
                # Get hybrid recommendations with specified alpha (balance between content similarity and recency)
                hybrid_recs = hybrid_recommendations(track_name, filtered_music_df, music_features_scaled, 
                                                   num_recommendations=num_recs, alpha=alpha)
                
                if not hybrid_recs.empty:
                    print(f"\nHybrid recommendations for '{track_name}':")
                    # Display a cleaner version of the recommendations
                    display_cols = ['Track Name', 'Artists', 'Album Name', 'Release Date', 'Popularity', 'Weighted Popularity', 'Hybrid Score']
                    display_cols = [col for col in display_cols if col in hybrid_recs.columns]
                    display(hybrid_recs[display_cols])
                    
                    # Optionally: Play a preview of the top recommendation if available
                    if 'Track ID' in hybrid_recs.columns:
                        top_track_id = hybrid_recs.iloc[0]['Track ID']
                        print(f"\nCheck out the top track on Spotify: https://open.spotify.com/track/{top_track_id}")
                else:
                    print("Could not generate recommendations.")
            else:
                print(f"'{track_name}' not found in the dataset. Try another track name.")
                print("\nSome available tracks:")
                display(filtered_music_df['Track Name'].sample(min(5, len(filtered_music_df))))
        else:
            print("Could not normalize music features for recommendations.")
    else:
        print("No music data available for recommendations.")

# Example usage:
# If you want to try with a specific track, modify this line:
input_track = "I'm Good (Blue)"  # Change to any track in your dataset
# Adjust the alpha value to control the balance between content similarity (higher alpha) and recency (lower alpha)
get_recommendations_for_track(input_track, alpha=0.6, num_recs=5)

In [17]:
def get_recommendations_with_retry(access_token, seed_tracks=None, seed_artists=None, seed_genres=None, limit=10, market='US'):
    """Get Spotify recommendations with robust error handling and retries"""
    sp = spotipy.Spotify(auth=access_token)
    
    try:
        # First check if we need to refresh the token
        try:
            # A simple API call to check if token is valid
            sp.recommendation_genre_seeds()
        except spotipy.exceptions.SpotifyException as e:
            if e.http_status == 401:
                print("Token expired. Refreshing...")
                # In a real application, you'd refresh the token here
                # For this notebook, we'll just notify the user
                print("Please run the token generation cell again to get a new access token.")
                return pd.DataFrame()
        
        # Try to get recommendations
        try:
            recommendations = sp.recommendations(
                seed_tracks=seed_tracks or [],
                seed_artists=seed_artists or [],
                seed_genres=seed_genres or [],
                limit=limit,
                market=market  # Add market parameter to avoid availability issues
            )
        except spotipy.exceptions.SpotifyException as e:
            # Handle case where the recommendation API itself failed
            if e.http_status == 404:
                print(f"Error getting recommendations: {e}")
                
                # If using genre seeds, check if they're valid
                if seed_genres:
                    valid_genres = get_available_genres(access_token)
                    invalid_genres = [g for g in seed_genres if g not in valid_genres]
                    if invalid_genres:
                        print(f"Invalid genre seeds: {invalid_genres}")
                        print(f"Valid genre options include: {', '.join(valid_genres[:10])}...")
                    
                    # Try with a fallback genre if possible
                    if valid_genres:
                        fallback_genres = [g for g in seed_genres if g in valid_genres]
                        if not fallback_genres and valid_genres:
                            fallback_genres = [valid_genres[0]]  # Use first valid genre
                            
                        if fallback_genres:
                            print(f"Retrying with valid genres: {fallback_genres}")
                            return get_recommendations_with_retry(
                                access_token, 
                                seed_tracks=seed_tracks,
                                seed_artists=seed_artists,
                                seed_genres=fallback_genres,
                                limit=limit
                            )
                
                return pd.DataFrame()
            else:
                print(f"Spotify API error: {e}")
                return pd.DataFrame()
        
        # Process recommendations
        recommended_tracks = []
        for track in recommendations['tracks']:
            track_info = {
                'Track Name': track['name'],
                'Artists': ', '.join([artist['name'] for artist in track['artists']]),
                'Album': track['album']['name'],
                'Release Date': track['album'].get('release_date'),
                'Popularity': track['popularity'],
                'Preview URL': track['preview_url'],
                'External URL': track['external_urls']['spotify'],
                'Track ID': track['id']
            }
            
            # Get audio features for the track (with error handling)
            try:
                audio_features = sp.audio_features(track['id'])[0]
                if audio_features:
                    track_info.update({
                        'Danceability': audio_features['danceability'],
                        'Energy': audio_features['energy'],
                        'Acousticness': audio_features['acousticness'],
                        'Valence': audio_features['valence'],  # Measures musical positiveness
                        'Speechiness': audio_features['speechiness'],
                        'Instrumentalness': audio_features['instrumentalness']
                    })
            except Exception as e:
                print(f"Error getting audio features for {track['name']}: {e}")
                
            recommended_tracks.append(track_info)
            
        return pd.DataFrame(recommended_tracks)
    except Exception as e:
        print(f"Unexpected error getting recommendations: {e}")
        return pd.DataFrame()